In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import time
import re
import pyodbc
from datetime import datetime

# ================== EXTRACT ==================
def extract(base_url, headers, max_pages=5):
    """
    Extract car data from PakWheels HTML pages using BeautifulSoup
    
    Args:
        base_url: PakWheels search page URL
        headers: Request headers
        max_pages: Maximum number of pages to scrape
        
    Returns:
        list: List of car dictionaries
    """
    all_cars = []
    
    try:
        for page in range(1, max_pages + 1):
            print(f"\n📄 Scraping page {page}...")
            
            # Construct URL for each page
            if page == 1:
                url = base_url
            else:
                url = f"{base_url}?page={page}"
            
            # Send request
            response = requests.get(url, headers=headers, timeout=15)
            response.raise_for_status()
            
            # Parse HTML with BeautifulSoup
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Find all car listings - they are in <li> tags with class "classified-listing"
            car_listings = soup.find_all('li', class_='classified-listing')
            
            if not car_listings:
                print(f"⚠ No more cars found on page {page}. Stopping.")
                break
            
            print(f"✓ Found {len(car_listings)} cars on page {page}")
            
            # Extract data from each car listing
            for car in car_listings:
                car_data = extract_car_details(car)
                if car_data:
                    all_cars.append(car_data)
            
            # Be polite - add delay between requests
            time.sleep(2)
        
        print(f"\n✓ Total cars extracted: {len(all_cars)}")
        return all_cars
        
    except requests.exceptions.RequestException as e:
        print(f"✗ Error during extraction: {e}")
        return all_cars


def extract_car_details(car_element):
    """
    Extract details from a single car listing element
    
    Args:
        car_element: BeautifulSoup element containing car info
        
    Returns:
        dict: Car details
    """
    try:
        car_data = {}
        
        # Extract car title/name
        title_elem = car_element.find('h3', class_='fs18')
        if not title_elem:
            title_elem = car_element.find('a', class_='car-name')
        car_data['title'] = title_elem.get_text(strip=True) if title_elem else 'N/A'
        
        # Extract price
        price_elem = car_element.find('div', class_='price-details')
        if not price_elem:
            price_elem = car_element.find('strong', string=re.compile(r'PKR'))
        car_data['price'] = price_elem.get_text(strip=True) if price_elem else 'N/A'
        
        # Extract location
        location_elem = car_element.find('ul', class_='list-unstyled')
        if location_elem:
            location_li = location_elem.find('li', class_='')
            if location_li:
                car_data['location'] = location_li.get_text(strip=True)
            else:
                car_data['location'] = 'N/A'
        else:
            car_data['location'] = 'N/A'
        
        # Extract specifications (year, mileage, engine, etc.)
        specs = car_element.find_all('li')
        year = 'N/A'
        mileage = 'N/A'
        engine = 'N/A'
        
        for spec in specs:
            spec_text = spec.get_text(strip=True)
            
            # Check for year (4 digit number)
            if re.search(r'\b(19|20)\d{2}\b', spec_text):
                year_match = re.search(r'\b(19|20)\d{2}\b', spec_text)
                if year_match:
                    year = year_match.group()
            
            # Check for mileage (contains 'km')
            if 'km' in spec_text.lower():
                mileage = spec_text
            
            # Check for engine info (contains 'cc' or engine type)
            if 'cc' in spec_text.lower() or 'petrol' in spec_text.lower() or 'diesel' in spec_text.lower():
                engine = spec_text
        
        car_data['year'] = year
        car_data['mileage'] = mileage
        car_data['engine'] = engine
        
        # Extract car link
        link_elem = car_element.find('a', href=True)
        if link_elem and link_elem.get('href'):
            href = link_elem['href']
            car_data['link'] = 'https://www.pakwheels.com' + href if href.startswith('/') else href
        else:
            car_data['link'] = 'N/A'
        
        # Extract posted date
        date_elem = car_element.find('div', class_='date')
        if not date_elem:
            date_elem = car_element.find('span', string=re.compile(r'(days?|hours?|weeks?)'))
        car_data['posted_date'] = date_elem.get_text(strip=True) if date_elem else 'N/A'
        
        # Check if featured
        featured = car_element.find('div', class_='featured-ribbon')
        car_data['featured'] = 'Yes' if featured else 'No'
        
        # Add scrape timestamp
        car_data['scraped_at'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        
        return car_data
        
    except Exception as e:
        print(f"⚠ Error extracting car details: {e}")
        return None


# ================== TRANSFORM ==================
def transform(cars_list):
    """
    Transform raw car data list into a cleaned pandas DataFrame
    
    Args:
        cars_list: List of car dictionaries
        
    Returns:
        pd.DataFrame: Transformed dataframe
    """
    try:
        if not cars_list:
            print("✗ No cars data to transform")
            return pd.DataFrame()
        
        # Convert to DataFrame
        df = pd.DataFrame(cars_list)
        
        # Data cleaning
        # Remove 'PKR' and commas from price, convert to numeric
        if 'price' in df.columns:
            df['price_clean'] = df['price'].str.replace('PKR', '', regex=False)
            df['price_clean'] = df['price_clean'].str.replace(',', '', regex=False)
            df['price_clean'] = df['price_clean'].str.strip()
            df['price_numeric'] = pd.to_numeric(df['price_clean'], errors='coerce')
            df = df.drop('price_clean', axis=1)
        
        # Extract year as numeric
        if 'year' in df.columns:
            df['year_numeric'] = pd.to_numeric(df['year'], errors='coerce')
        
        # Extract mileage as numeric (remove 'km' and commas)
        if 'mileage' in df.columns:
            df['mileage_clean'] = df['mileage'].str.replace('km', '', regex=False)
            df['mileage_clean'] = df['mileage_clean'].str.replace(',', '', regex=False)
            df['mileage_clean'] = df['mileage_clean'].str.strip()
            df['mileage_numeric'] = pd.to_numeric(df['mileage_clean'], errors='coerce')
            df = df.drop('mileage_clean', axis=1)
        
        # Remove duplicates based on title and price
        df = df.drop_duplicates(subset=['title', 'price'], keep='first')
        
        print(f"✓ Data transformed. Shape: {df.shape}")
        print(f"✓ Columns: {list(df.columns)}")
        
        return df
        
    except Exception as e:
        print(f"✗ Error transforming data: {e}")
        return pd.DataFrame()


# ================== LOAD ==================
def load_to_csv(df, filename='pakwheels_karachi_cars.csv'):
    """
    Load DataFrame to CSV file
    
    Args:
        df: pandas DataFrame
        filename: Output CSV filename
        
    Returns:
        bool: Success status
    """
    try:
        if df.empty:
            print("✗ DataFrame is empty. Nothing to save.")
            return False
        
        # Save to CSV
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"✓ Data loaded successfully to '{filename}'")
        print(f"✓ Total records saved: {len(df)}")
        
        return True
        
    except Exception as e:
        print(f"✗ Error loading data to CSV: {e}")
        return False


def load_to_sql(df, connection_string, table_name='pakwheels_cars'):
    """
    Load DataFrame to SQL Server database
    
    Args:
        df: pandas DataFrame
        connection_string: SQL Server connection string
        table_name: Name of the table to create/insert into
        
    Returns:
        bool: Success status
    """
    try:
        if df.empty:
            print("✗ DataFrame is empty. Nothing to load to SQL.")
            return False
        
        print(f"\n📊 Connecting to SQL Server...")
        
        # Connect to SQL Server
        conn = pyodbc.connect(connection_string)
        cursor = conn.cursor()
        
        print(f"✓ Connected to SQL Server successfully!")
        
        # Create table if it doesn't exist
        create_table_query = f"""
        IF NOT EXISTS (SELECT * FROM sysobjects WHERE name='{table_name}' AND xtype='U')
        CREATE TABLE {table_name} (
            id INT IDENTITY(1,1) PRIMARY KEY,
            title NVARCHAR(500),
            price NVARCHAR(100),
            location NVARCHAR(200),
            year NVARCHAR(50),
            mileage NVARCHAR(100),
            engine NVARCHAR(200),
            link NVARCHAR(500),
            posted_date NVARCHAR(100),
            featured NVARCHAR(10),
            price_numeric FLOAT,
            year_numeric INT,
            mileage_numeric FLOAT,
            scraped_at DATETIME
        )
        """
        cursor.execute(create_table_query)
        conn.commit()
        print(f"✓ Table '{table_name}' ready!")
        
        # Insert data
        print(f"📥 Inserting {len(df)} records...")
        
        insert_query = f"""
        INSERT INTO {table_name} 
        (title, price, location, year, mileage, engine, link, posted_date, featured, 
         price_numeric, year_numeric, mileage_numeric, scraped_at)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """
        
        for index, row in df.iterrows():
            cursor.execute(insert_query, 
                          row['title'], row['price'], row['location'], 
                          row['year'], row['mileage'], row['engine'],
                          row['link'], row['posted_date'], row['featured'],
                          row.get('price_numeric'), row.get('year_numeric'), 
                          row.get('mileage_numeric'), row['scraped_at'])
        
        conn.commit()
        print(f"✓ Successfully inserted {len(df)} records into SQL Server!")
        
        # Display statistics
        if 'price_numeric' in df.columns:
            valid_prices = df['price_numeric'].dropna()
            if len(valid_prices) > 0:
                print(f"\n📊 Price Statistics:")
                print(f"   Average Price: PKR {valid_prices.mean():,.0f}")
                print(f"   Min Price: PKR {valid_prices.min():,.0f}")
                print(f"   Max Price: PKR {valid_prices.max():,.0f}")
        
        cursor.close()
        conn.close()
        
        return True
        
    except pyodbc.Error as e:
        print(f"✗ SQL Server Error: {e}")
        return False
    except Exception as e:
        print(f"✗ Error loading data to SQL: {e}")
        return False


# ================== MAIN ETL PIPELINE ==================
def main():
    """
    Main ETL pipeline for PakWheels data using BeautifulSoup
    """
    print("="*60)
    print("PakWheels Karachi Used Cars - ETL Pipeline")
    print("="*60)
    
    # Configuration
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"
    }
    
    # PakWheels Karachi used cars search page
    base_url = "https://www.pakwheels.com/used-cars/search/-/ct_karachi/"
    
    max_pages = 5  # Number of pages to scrape
    
    # SQL Server Connection String
    # MODIFY THIS WITH YOUR SQL SERVER DETAILS
    connection_string = (
        "DRIVER={ODBC Driver 17 for SQL Server};"
        "SERVER=DESKTOP-PI1U0J8;"  # Change to your server name
        "DATABASE=PakWheelsDB;"           # Change to your database name
        "Trusted_Connection=yes;"         # For Windows Authentication
        # For SQL Authentication, use:
        # "UID=your_username;"
        # "PWD=your_password;"
    )
    
    # ETL Process
    print(f"\n[1/3] EXTRACT - Scraping data using BeautifulSoup...")
    print(f"🌐 Target: {base_url}")
    print(f"📄 Pages to scrape: {max_pages}")
    
    cars_data = extract(base_url, headers, max_pages)
    
    if cars_data:
        print(f"\n[2/3] TRANSFORM - Converting to DataFrame and cleaning...")
        df = transform(cars_data)
        
        if not df.empty:
            # Display sample data
            print("\n📋 Sample Data (First 5 rows):")
            print(df[['title', 'price', 'year', 'location']].head())
            
            print(f"\n[3/3] LOAD - Saving data...")
            
            # Load to CSV
            print("\n💾 Saving to CSV...")
            load_to_csv(df, 'pakwheels_karachi_cars.csv')
            
            # Load to SQL Server
            print("\n💾 Saving to SQL Server...")
            load_to_sql(df, connection_string, 'pakwheels_cars')
            
        else:
            print("\n✗ Transform failed. Skipping load step.")
    else:
        print("\n✗ Extract failed. Pipeline terminated.")
    
    print("\n" + "="*60)
    print("ETL Pipeline Complete!")
    print("="*60)


# Run the pipeline
if __name__ == "__main__":
    main()

PakWheels Karachi Used Cars - ETL Pipeline

[1/3] EXTRACT - Scraping data using BeautifulSoup...
🌐 Target: https://www.pakwheels.com/used-cars/search/-/ct_karachi/
📄 Pages to scrape: 5

📄 Scraping page 1...
✓ Found 25 cars on page 1

📄 Scraping page 2...
✓ Found 25 cars on page 2

📄 Scraping page 3...
✓ Found 27 cars on page 3

📄 Scraping page 4...
✓ Found 28 cars on page 4

📄 Scraping page 5...
✓ Found 28 cars on page 5

✓ Total cars extracted: 133

[2/3] TRANSFORM - Converting to DataFrame and cleaning...
✓ Data transformed. Shape: (124, 13)
✓ Columns: ['title', 'price', 'location', 'year', 'mileage', 'engine', 'link', 'posted_date', 'featured', 'scraped_at', 'price_numeric', 'year_numeric', 'mileage_numeric']

📋 Sample Data (First 5 rows):
                                     title         price  year location
0            Toyota Raize  2020 Z for Sale  PKR 61.5lacs  2020     2020
1           Suzuki Alto  2024 VXR for Sale  PKR 27.5lacs  2024     2024
2   Toyota Vitz  2013 F 1.0 for